# Notebook 16: MechInt Pipeline & Database Infrastructure

**Duration**: 45-60 minutes  
**Prerequisites**: Notebooks 01-10 (basic familiarity), Python 3.8+, PyTorch

## Learning Objectives

By the end of this notebook, you will:

1. Understand the **MechIntDatabase** hybrid storage architecture (HDF5 + SQLite)
2. Store and query mechanistic interpretability results efficiently
3. Use **MechIntPipeline** to run comprehensive analysis workflows
4. Configure pipeline modes (quick, standard, comprehensive)
5. Implement custom analysis stages and generate reports
6. Leverage content-based deduplication and result caching

## Why Pipeline and Database Infrastructure?

Mechanistic interpretability research generates large amounts of heterogeneous data:

- **Circuits**: Graphs with nodes, edges, importance scores
- **Activation patterns**: High-dimensional tensors
- **Thermodynamic quantities**: Time series, energy flows, entropy production
- **Dynamics**: Trajectories, flow fields, phase space structures

Challenges:
- **Storage**: Efficient storage of mixed data types (arrays, metadata, graphs)
- **Retrieval**: Fast queries across experiments
- **Reproducibility**: Content-based hashing prevents duplicate computation
- **Workflow**: Coordinating multiple analysis stages

**Solution**: `MechIntDatabase` + `MechIntPipeline`

### Architecture

**MechIntDatabase**:
- HDF5 for large arrays (activations, trajectories)
- SQLite for metadata and fast queries
- Content-based hashing (SHA256) for deduplication
- Tagging system for experiment organization

**MechIntPipeline**:
- Configurable analysis stages
- Three preset modes: quick, standard, comprehensive
- Checkpointing and recovery
- Automatic report generation

## Key References

Database design:
- HDF5 documentation: https://www.hdfgroup.org/solutions/hdf5/
- SQLite documentation: https://www.sqlite.org/docs.html

Workflow systems:
- Prefect: https://www.prefect.io/
- Luigi: https://luigi.readthedocs.io/

Content-addressable storage:
- Git internals: https://git-scm.com/book/en/v2/Git-Internals-Plumbing-and-Porcelain

## Setup

In [ ]:
import sys
import tempfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Dict, List, Optional, Any

# Import database and pipeline
from neuros_mechint.database import MechIntDatabase
from neuros_mechint.pipeline import MechIntPipeline, PipelineConfig

# Import result types (including new extended types)
from neuros_mechint.results import (
    MechIntResult,
    CircuitResult,
    DynamicsResult,
    InformationResult,
    AlignmentResult,
    FractalResult,
    ResultCollection
)
from neuros_mechint.results_extended import (
    BiophysicalResult,
    InterventionResult,
    CriticalityResult,
    MultifractalResult
)

# Import visualization
from neuros_mechint.visualization import EnhancedVisualizer

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Matplotlib style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

## Part 1: MechIntDatabase Basics

The database uses a hybrid architecture:
- **SQLite**: Fast queries on metadata (tags, timestamps, result types)
- **HDF5**: Efficient storage of large arrays
- **Content hashing**: SHA256 hash of inputs prevents duplicate computation

### 1.1 Database Setup and Storage

In [ ]:
# Create temporary database for demonstration
temp_dir = tempfile.mkdtemp()
db = MechIntDatabase(root_dir=temp_dir)

print(f"Database created at: {temp_dir}")
print(f"SQLite database: {Path(temp_dir) / 'mechint.db'}")
print(f"HDF5 storage: {Path(temp_dir) / 'results.h5'}")

### 1.2 Storing Analysis Results

Let's run some analyses and store the results.

In [ ]:
# Create a simple model for demonstration
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim: int = 20, hidden_dim: int = 32, output_dim: int = 3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

model = SimpleClassifier().to(device)
model.eval()

# Generate sample data
batch_size = 32
input_dim = 20
sample_inputs = torch.randn(batch_size, input_dim, device=device)
sample_labels = torch.randint(0, 3, (batch_size,), device=device)

print(f"Model: {model.__class__.__name__}")
print(f"Sample inputs: {sample_inputs.shape}")

In [ ]:
# Store a simple custom result to demonstrate database functionality
print("Creating and storing example results...")

# Create a circuit result manually (simpler than running full ACDC)
circuit_result = CircuitResult(
    method="ACDC",
    data={
        'nodes': ['layer1.0', 'layer1.5', 'layer2.0', 'layer2.5'],
        'edges': [('layer1.0', 'layer2.0'), ('layer1.5', 'layer2.5')],
        'importance_scores': {'layer1.0': 0.8, 'layer1.5': 0.6, 'layer2.0': 0.7, 'layer2.5': 0.5}
    },
    metadata={'model': 'simple_classifier', 'threshold': 0.05},
    metrics={'n_nodes': 4, 'n_edges': 2, 'iterations': 10}
)

# Store in database with tags
circuit_id = db.store(
    result=circuit_result,
    tags=['acdc', 'simple_classifier', 'experiment_1']
)

print(f"✓ Circuit stored with ID: {circuit_id}")
print(f"  - Nodes: {len(circuit_result.data['nodes'])}")
print(f"  - Edges: {len(circuit_result.data['edges'])}")
print(f"  - Tags: {db.get_tags(circuit_id)}")

In [ ]:
# Demonstrate storing new result types (biophysical, interventions, criticality, multifractal)
print("\nDemonstrating new result types...")

# BiophysicalResult
biophysical_result = BiophysicalResult(
    method="HodgkinHuxley",
    data={'simulation_time': np.arange(0, 100, 0.1)},
    metadata={'model': 'simple_classifier', 'neuron_type': 'pyramidal'},
    metrics={'spike_count': 15, 'mean_isi': 6.7},
    voltages=np.random.randn(1000, 3) * 10 - 70,  # 3 compartments
    spike_times=[np.array([10, 20, 30]), np.array([15, 25, 35])],
    spike_rate=15.0,
    atp_levels=np.random.rand(1000) * 2.5 + 1.0
)

biophys_id = db.store(
    result=biophysical_result,
    tags=['biophysical', 'simple_classifier', 'experiment_1']
)

print(f"✓ Biophysical analysis stored with ID: {biophys_id}")
print(f"  - Spike rate: {biophysical_result.spike_rate:.1f} Hz")
print(f"  - Mean ATP: {biophysical_result.atp_levels.mean():.2f} mM")

# CriticalityResult
criticality_result = CriticalityResult(
    method="AvalancheAnalysis",
    data={'avalanche_sizes': np.random.power(1.5, 100)},
    metadata={'model': 'simple_classifier'},
    metrics={'branching_param': 0.98, 'power_law_exp': 1.52},
    branching_parameter=0.98,
    n_avalanches=100,
    size_distribution=np.random.power(1.5, 100),
    duration_distribution=np.random.exponential(5, 100),
    is_critical=True
)

crit_id = db.store(
    result=criticality_result,
    tags=['criticality', 'simple_classifier', 'experiment_1']
)

print(f"✓ Criticality analysis stored with ID: {crit_id}")
print(f"  - Branching parameter: {criticality_result.branching_parameter:.3f}")
print(f"  - Is critical: {criticality_result.is_critical}")
print(f"  - Tags: {db.get_tags(crit_id)}")

### 1.3 Content-Based Deduplication

The database uses content hashing to detect duplicate computations.

In [ ]:
# Try to store the same circuit result again
print("Attempting to store duplicate result...")
duplicate_id = db.store(
    result=circuit_result,
    tags=['acdc', 'duplicate_test']
)

print(f"Original ID:  {circuit_id}")
print(f"Duplicate ID: {duplicate_id}")
print(f"Same? {circuit_id == duplicate_id}")

# Tags are merged for duplicate content
print(f"\nMerged tags: {db.get_tags(circuit_id)}")

### 1.4 Querying Results

The database supports tag-based queries and retrieval.

In [ ]:
# Query by tag
print("Querying all 'experiment_1' results:")
exp1_ids = db.query(tags=['experiment_1'])
print(f"Found {len(exp1_ids)} results")

for result_id in exp1_ids:
    result = db.load(result_id)
    print(f"  - {result_id}: {type(result).__name__}")

# Query thermodynamic results
print("\nQuerying 'thermodynamics' results:")
thermo_ids = db.query(tags=['thermodynamics'])
print(f"Found {len(thermo_ids)} results")

# Query with multiple tags (AND logic)
print("\nQuerying 'simple_classifier' AND 'acdc' results:")
classifier_acdc_ids = db.query(tags=['simple_classifier', 'acdc'])
print(f"Found {len(classifier_acdc_ids)} results")

### 1.5 Database Statistics

In [ ]:
# Get database statistics
all_ids = db.list_all()
print(f"Total stored results: {len(all_ids)}")

# Group by result type
from collections import defaultdict
type_counts = defaultdict(int)

for result_id in all_ids:
    result = db.load(result_id)
    type_counts[type(result).__name__] += 1

print("\nResults by type:")
for result_type, count in sorted(type_counts.items()):
    print(f"  {result_type}: {count}")

# Get all unique tags
all_tags = set()
for result_id in all_ids:
    all_tags.update(db.get_tags(result_id))

print(f"\nUnique tags: {sorted(all_tags)}")

## Part 2: MechIntPipeline Configuration

The pipeline orchestrates multi-stage analysis workflows.

### Pipeline Modes

1. **Quick** (~5 min): Basic circuit discovery + energy analysis
2. **Standard** (~15 min): + Thermodynamics + Dynamics
3. **Comprehensive** (~30+ min): + Advanced comparisons + Full visualization

### 2.1 Quick Pipeline Mode

In [ ]:
# Configure quick pipeline
quick_config = PipelineConfig(
    depth='quick',
    enabled_analyses={'sae', 'circuits'},  # Minimal set for quick analysis
    parallel=True,
    use_cache=True,
    verbose=True
)

# Create pipeline (pass model and db path)
quick_pipeline = MechIntPipeline(
    model=model,
    db_path=temp_dir,
    config=quick_config,
    device=device
)

print("Quick Pipeline Configuration:")
print(f"  Depth: {quick_config.depth}")
print(f"  Enabled analyses: {sorted(quick_config.enabled_analyses)}")
print(f"  Parallel execution: {quick_config.parallel}")
print(f"  Cache enabled: {quick_config.use_cache}")

In [ ]:
# Run quick pipeline
print("Running quick pipeline...")
quick_results = quick_pipeline.run(
    inputs=sample_inputs,
    experiment_tags=['quick_demo']
)

print(f"\n✓ Quick pipeline complete!")
print(f"Results: {list(quick_results.keys())}")
print(f"\nAnalyses completed:")
for analysis_name in quick_results:
    print(f"  - {analysis_name}")

### 2.2 Standard Pipeline Mode

In [ ]:
# Configure standard pipeline (more comprehensive)
standard_config = PipelineConfig(
    depth='standard',
    enabled_analyses={'sae', 'circuits', 'dynamics', 'fractals', 'biophysical'},
    parallel=True,
    use_cache=True,
    verbose=True
)

standard_pipeline = MechIntPipeline(
    model=model,
    db_path=temp_dir,
    config=standard_config,
    device=device
)

print("Standard Pipeline Configuration:")
print(f"  Depth: {standard_config.depth}")
print(f"  Enabled analyses: {sorted(standard_config.enabled_analyses)}")
print(f"  Number of analyses: {len(standard_config.enabled_analyses)}")

In [ ]:
# Run standard pipeline
print("Running standard pipeline...")
standard_results = standard_pipeline.run(
    inputs=sample_inputs,
    experiment_tags=['standard_demo']
)

print(f"\n✓ Standard pipeline complete!")
print(f"Results: {list(standard_results.keys())}")
print(f"\nAdditional stages vs quick:")
new_stages = set(standard_results.keys()) - set(quick_results.keys())
print(f"  {new_stages}")

### 2.3 Custom Pipeline Configuration

You can create custom pipelines with specific analysis stages.

In [ ]:
# Custom configuration: Biophysical + Criticality + Multifractal
custom_config = PipelineConfig(
    depth='custom',
    enabled_analyses={'biophysical', 'criticality', 'multifractal'},
    parallel=True,
    use_cache=True,
    verbose=True
)

custom_pipeline = MechIntPipeline(
    model=model,
    db_path=temp_dir,
    config=custom_config,
    device=device
)

print("Custom Pipeline - Biophysical + Criticality + Multifractal:")
custom_results = custom_pipeline.run(
    inputs=sample_inputs,
    experiment_tags=['custom_advanced']
)

print(f"Results: {list(custom_results.keys())}")

## Part 3: End-to-End Workflow

Let's demonstrate a complete research workflow:
1. Train multiple model variants
2. Run comprehensive analysis pipeline
3. Query and compare results
4. Generate comparative visualizations

### 3.1 Create Model Variants

In [ ]:
# Create three model variants: Wide, Deep, Balanced
models = {
    'wide': SimpleClassifier(input_dim=20, hidden_dim=64, output_dim=3).to(device),
    'deep': nn.Sequential(
        nn.Linear(20, 24),
        nn.ReLU(),
        nn.Linear(24, 24),
        nn.ReLU(),
        nn.Linear(24, 24),
        nn.ReLU(),
        nn.Linear(24, 24),
        nn.ReLU(),
        nn.Linear(24, 3)
    ).to(device),
    'balanced': SimpleClassifier(input_dim=20, hidden_dim=32, output_dim=3).to(device)
}

for name, model in models.items():
    model.eval()
    param_count = sum(p.numel() for p in model.parameters())
    print(f"{name:10s}: {param_count:,} parameters")

### 3.2 Run Comprehensive Analysis on All Models

In [ ]:
# Use comprehensive pipeline configuration
comprehensive_config = PipelineConfig(
    depth='comprehensive',
    enabled_analyses={
        'sae', 'circuits', 'dynamics', 'fractals',
        'biophysical', 'interventions', 'criticality', 
        'multifractal', 'temporal', 'alignment'
    },
    parallel=True,
    use_cache=True,
    verbose=True,
    auto_report=True  # Auto-generate report
)

# Store results for each model
model_results = {}

for model_name, model_instance in models.items():
    print(f"\nAnalyzing {model_name} model...")
    
    # Create pipeline for this model
    pipeline = MechIntPipeline(
        model=model_instance,
        db_path=temp_dir,
        config=comprehensive_config,
        device=device
    )
    
    # Run analysis
    results = pipeline.run(
        inputs=sample_inputs,
        experiment_tags=['comparison', model_name]
    )
    
    model_results[model_name] = results
    print(f"  ✓ {len(results)} stages completed")

print("\n" + "="*50)
print("All models analyzed!")
print(f"Total results in database: {len(db.list_all())}")

### 3.3 Query and Compare Results

In [ ]:
# Query and compare biophysical and criticality results
print("Comparing biophysical and criticality results across models...\n")

# Query biophysical results
biophys_ids = db.query(tags=['biophysical'])
crit_ids = db.query(tags=['criticality'])

print(f"Found {len(biophys_ids)} biophysical analyses")
print(f"Found {len(crit_ids)} criticality analyses")

# Compare metrics across models
comparison_data = {}

# Group results by model
for result_id in biophys_ids:
    result = db.load(result_id)
    tags = db.get_tags(result_id)
    
    # Extract model name from tags
    model_tags = [t for t in tags if t in ['wide', 'deep', 'balanced']]
    if model_tags:
        model_name = model_tags[0]
        if model_name not in comparison_data:
            comparison_data[model_name] = {}
        
        if isinstance(result, BiophysicalResult):
            comparison_data[model_name]['spike_rate'] = result.spike_rate or 0.0
            comparison_data[model_name]['atp_mean'] = result.atp_levels.mean() if result.atp_levels is not None else 0.0

for result_id in crit_ids:
    result = db.load(result_id)
    tags = db.get_tags(result_id)
    
    model_tags = [t for t in tags if t in ['wide', 'deep', 'balanced']]
    if model_tags:
        model_name = model_tags[0]
        if model_name not in comparison_data:
            comparison_data[model_name] = {}
        
        if isinstance(result, CriticalityResult):
            comparison_data[model_name]['branching_param'] = result.branching_parameter
            comparison_data[model_name]['is_critical'] = result.is_critical
            comparison_data[model_name]['n_avalanches'] = result.n_avalanches

print("\nModel Comparison:")
print(f"{'Model':<12} {'Spike Rate (Hz)':>15} {'ATP (mM)':>12} {'σ (branching)':>15} {'Critical?':>10}")
print("-" * 75)
for model_name, metrics in sorted(comparison_data.items()):
    spike_rate = metrics.get('spike_rate', 0)
    atp = metrics.get('atp_mean', 0)
    branch = metrics.get('branching_param', 0)
    critical = 'Yes' if metrics.get('is_critical', False) else 'No'
    print(f"{model_name:<12} {spike_rate:>15.2f} {atp:>12.2f} {branch:>15.3f} {critical:>10s}")

### 3.4 Visualize Comparative Results

In [ ]:
# Extract circuit and biophysical metrics for visualization
circuit_ids = db.query(tags=['circuits'])
circuit_comparison = {}

for result_id in circuit_ids:
    result = db.load(result_id)
    tags = db.get_tags(result_id)
    
    model_tags = [t for t in tags if t in ['wide', 'deep', 'balanced']]
    if model_tags:
        model_name = model_tags[0]
        if isinstance(result, CircuitResult):
            circuit_comparison[model_name] = {
                'nodes': len(result.data.get('nodes', [])),
                'edges': len(result.data.get('edges', [])),
                'complexity': result.metrics.get('complexity', 0)
            }

# Create comparative visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Model Comparison: New Analyses', fontsize=14, fontweight='bold')

# Plot 1: Spike Rate (from biophysical analysis)
if comparison_data:
    ax = axes[0, 0]
    models_list = list(comparison_data.keys())
    spike_rates = [comparison_data[m].get('spike_rate', 0) for m in models_list]
    ax.bar(models_list, spike_rates, alpha=0.8, color='steelblue')
    ax.set_xlabel('Model')
    ax.set_ylabel('Spike Rate (Hz)')
    ax.set_title('Neural Activity (Biophysical Analysis)')
    ax.grid(True, alpha=0.3, axis='y')

# Plot 2: ATP Levels (metabolic efficiency)
if comparison_data:
    ax = axes[0, 1]
    atp_levels = [comparison_data[m].get('atp_mean', 0) for m in models_list]
    colors = ['green' if a > 2.0 else 'orange' if a > 1.5 else 'red' for a in atp_levels]
    ax.bar(models_list, atp_levels, alpha=0.8, color=colors)
    ax.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5, label='Healthy threshold')
    ax.set_xlabel('Model')
    ax.set_ylabel('ATP Concentration (mM)')
    ax.set_title('Metabolic State')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Branching Parameter (criticality)
if comparison_data:
    ax = axes[1, 0]
    branch_params = [comparison_data[m].get('branching_param', 0) for m in models_list]
    distance_from_critical = [abs(b - 1.0) for b in branch_params]
    colors_br = ['green' if d < 0.1 else 'orange' if d < 0.2 else 'red' for d in distance_from_critical]
    ax.bar(models_list, branch_params, alpha=0.8, color=colors_br)
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Critical (σ=1)')
    ax.set_xlabel('Model')
    ax.set_ylabel('Branching Parameter σ')
    ax.set_title('Criticality Analysis')
    ax.set_ylim([0.5, 1.5])
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

# Plot 4: Multi-metric radar
if circuit_comparison:
    ax = axes[1, 1]
    models_list_circ = list(circuit_comparison.keys())
    nodes = [circuit_comparison[m]['nodes'] for m in models_list_circ]
    edges = [circuit_comparison[m]['edges'] for m in models_list_circ]
    
    x = np.arange(len(models_list_circ))
    width = 0.35
    ax.bar(x - width/2, nodes, width, label='Circuit Nodes', alpha=0.8)
    ax.bar(x + width/2, edges, width, label='Circuit Edges', alpha=0.8)
    ax.set_xlabel('Model')
    ax.set_ylabel('Count')
    ax.set_title('Circuit Complexity')
    ax.set_xticks(x)
    ax.set_xticklabels(models_list_circ)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nInsights:")
if comparison_data:
    best_atp_idx = np.argmax([comparison_data[m].get('atp_mean', 0) for m in models_list])
    most_critical_idx = np.argmin([abs(comparison_data[m].get('branching_param', 0) - 1.0) for m in models_list])
    print(f"  - Best metabolic efficiency: {models_list[best_atp_idx]}")
    print(f"  - Closest to criticality: {models_list[most_critical_idx]}")

## Part 4: Advanced Features

### 4.1 Checkpoint and Recovery

Long-running pipelines can be interrupted and resumed.

In [ ]:
# Pipeline automatically caches results in the database
# Results can be retrieved and reused
print("Demonstrating result caching and reuse...")

# Query previously computed results
experiment_name = 'standard_demo'
prev_results = db.query(tags=[experiment_name])

print(f"Found {len(prev_results)} cached results for '{experiment_name}'")
print("\nCached analyses:")
for result_id in prev_results[:5]:  # Show first 5
    result = db.load(result_id)
    print(f"  - {result.method}: {type(result).__name__}")

# Database provides automatic deduplication
# Running the same analysis again will return cached results
print("\nDatabase provides automatic content-based caching")
print("Running identical analysis will reuse existing results")

### 4.2 Pipeline Report Generation

Generate a comprehensive report of pipeline results.

In [ ]:
# Generate summary report from results
print("Pipeline Analysis Summary Report")
print("=" * 60)

if model_results.get('balanced'):
    balanced_results = model_results['balanced']
    
    print(f"\nModel: Balanced Classifier")
    print(f"Analyses completed: {len(balanced_results)}")
    print("\nResults summary:")
    
    for analysis_name, result in balanced_results.items():
        print(f"\n{analysis_name.upper()}:")
        if hasattr(result, 'metrics'):
            for metric_name, metric_value in result.metrics.items():
                print(f"  - {metric_name}: {metric_value}")

print("\n" + "=" * 60)
print("Analysis complete. Results stored in database.")
print(f"Database location: {db.root_dir}")
print(f"Total results: {len(db.list_all())}")

### 4.3 Custom Analysis Stages

You can add custom analysis stages to the pipeline.

In [ ]:
# Define a custom analysis function
@dataclass
class CustomMetricResult:
    """Custom analysis result."""
    metric_name: str
    value: float
    metadata: Dict[str, Any]

def custom_sparsity_analysis(
    model: nn.Module,
    inputs: torch.Tensor,
    device: torch.device
) -> CustomMetricResult:
    """Analyze activation sparsity in the model."""
    activations = []
    
    def hook(module, input, output):
        if isinstance(output, torch.Tensor):
            activations.append(output.detach())
    
    # Register hooks
    hooks = []
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            hooks.append(module.register_forward_hook(hook))
    
    # Forward pass
    with torch.no_grad():
        model(inputs)
    
    # Remove hooks
    for h in hooks:
        h.remove()
    
    # Calculate sparsity (fraction of zeros)
    if activations:
        all_activations = torch.cat([a.flatten() for a in activations])
        sparsity = (all_activations == 0).float().mean().item()
    else:
        sparsity = 0.0
    
    return CustomMetricResult(
        metric_name='activation_sparsity',
        value=sparsity,
        metadata={
            'n_relu_layers': len(activations),
            'total_activations': sum(a.numel() for a in activations)
        }
    )

# Run custom analysis
print("Running custom sparsity analysis...")
sparsity_result = custom_sparsity_analysis(
    model=models['balanced'],
    inputs=sample_inputs,
    device=device
)

# Store in database
sparsity_id = db.store(
    result=sparsity_result,
    tags=['custom_analysis', 'sparsity', 'balanced']
)

print(f"\n✓ Custom analysis complete")
print(f"  Metric: {sparsity_result.metric_name}")
print(f"  Value: {sparsity_result.value:.3f}")
print(f"  ReLU layers analyzed: {sparsity_result.metadata['n_relu_layers']}")
print(f"  Stored with ID: {sparsity_id}")

## Summary

In this notebook, we explored the **MechIntDatabase** and **MechIntPipeline** infrastructure:

### Key Concepts

1. **Hybrid Storage**:
   - SQLite: Fast metadata queries
   - HDF5: Efficient array storage
   - Content hashing: Automatic deduplication

2. **Tag-Based Organization**:
   ```python
   db.store(result, tags=['experiment', 'acdc', 'model_v1'])
   results = db.query(tags=['experiment', 'acdc'])  # AND logic
   ```

3. **Pipeline Modes**:
   - Quick: Fast iteration (~5 min)
   - Standard: Balanced analysis (~15 min)
   - Comprehensive: Full analysis (~30+ min)
   - Custom: User-defined stages

4. **Workflow Benefits**:
   - Reproducibility: Content-based deduplication
   - Efficiency: Checkpoint and resume
   - Organization: Tag-based retrieval
   - Extensibility: Custom analysis stages

### When to Use Each Component

**Use MechIntDatabase when**:
- Running multiple experiments
- Need to compare results across models
- Want automatic deduplication
- Require fast queries on metadata

**Use MechIntPipeline when**:
- Running standardized analysis workflows
- Need checkpoint/resume capability
- Want automated report generation
- Coordinating multiple analysis stages

### Best Practices

1. **Tag Strategically**: Use hierarchical tags (e.g., `experiment_name`, `model_type`, `analysis_type`)
2. **Enable Checkpoints**: For long-running pipelines
3. **Query Before Compute**: Check if result already exists
4. **Custom Stages**: Extend pipeline with domain-specific analyses
5. **Regular Cleanup**: Archive old experiments to manage storage

## Exercises

### Exercise 1: Custom Pipeline for Sparse Models

Create a custom pipeline configuration that focuses on sparsity analysis:
1. Disable thermodynamics and dynamics
2. Enable circuits and energy
3. Add custom sparsity analysis stage
4. Run on a model with L1 regularization
5. Compare sparsity vs energy cost

**Starter code:**

In [ ]:
# Implement sparse model pipeline
def create_sparse_pipeline(db, device):
    # TODO: Create custom config for sparsity-focused analysis
    config = PipelineConfig(
        mode='custom',
        # Add appropriate flags
    )
    # TODO: Return configured pipeline
    pass

# TODO: Create sparse model with L1 regularization
# TODO: Run pipeline
# TODO: Compare sparsity vs energy efficiency

### Exercise 2: Database Query Optimization

Implement efficient queries for a large-scale experiment:
1. Store results from 20+ model variants
2. Query all results from a specific date range
3. Find top-5 most efficient models (circuit nodes / energy)
4. Generate comparative report

**Starter code:**

In [ ]:
# Implement efficient querying
def find_top_efficient_models(db, n_top=5):
    # TODO: Query all circuit and energy results
    # TODO: Calculate efficiency metric
    # TODO: Return top N models with metadata
    pass

# TODO: Implement date range filtering
# TODO: Generate comparative visualization

### Exercise 3: Batch Analysis Pipeline

Create a pipeline that analyzes multiple models in parallel:
1. Define 5 model architectures (varying depth/width)
2. Run comprehensive pipeline on each
3. Store all results with consistent tagging
4. Query and visualize trade-offs:
   - Model complexity vs circuit complexity
   - Parameter count vs energy cost
   - Reversibility vs accuracy

**Starter code:**

In [ ]:
# Implement batch analysis
def batch_analyze_models(models_dict, pipeline, db, inputs, targets):
    # TODO: Run pipeline on each model
    # TODO: Store with consistent tags (include timestamp, architecture)
    # TODO: Return aggregated results
    pass

# TODO: Create 5 model variants
# TODO: Run batch analysis
# TODO: Query results
# TODO: Create multi-dimensional comparison plots

## Next Steps

You've completed the Phase 2 example notebooks! Here's what to explore next:

1. **Integration**: Combine techniques from multiple notebooks
2. **Real Models**: Apply to production architectures (ResNet, Transformers, etc.)
3. **Custom Analyzers**: Implement domain-specific interpretability methods
4. **Visualization**: Create interactive Bokeh dashboards for exploration
5. **Research**: Use infrastructure for novel mechanistic interpretability research

Recommended reading order:
- Notebooks 01-05: Core foundations
- Notebooks 06-10: Advanced SAE analysis  
- Notebooks 11-13: Circuit discovery
- Notebooks 14-15: Dynamics and energy
- **Notebook 16** (this one): Infrastructure

For comprehensive Phase 2 guidance, see `PHASE2_GUIDE.md` in the examples directory.